# **Feature Hashing**
*aka the hashing trick · dimensionality technique in ML*

---

## **Motivation**

- Categorical features with very large or unknown vocabulary (e.g. URLs, user IDs, raw text tokens) are expensive to one-hot encode.
- Vocabulary size isn't always known ahead of time — new categories can appear at inference.
- Storing a full feature dictionary consumes significant memory; hashing avoids storing it entirely.
- ***Streaming / online learning:*** no need to precompute the vocabulary over the whole dataset first.
- Useful when the model must be deployed in memory-constrained environments (mobile, edge).

---

## **Introduction**

- Apply a hash function to map any feature string to a fixed-size vector of length $m$:

$$h : \mathcal{F} \rightarrow \{0, 1, \dots, m-1\}$$

- Result is a sparse vector $\phi \in \mathbb{R}^m$: slot $h(x)$ gets value $1$ (or a weight); all others are $0$:

$$\phi(x_1, x_2, \dots, x_n)_i = \sum_{x_j \,:\, h(x_j) = i} 1$$

- Multiple features can map to the same bucket — this is a **collision**.
- A second sign hash $\xi : \mathcal{F} \rightarrow \{+1, -1\}$ is applied to the value to make collisions unbiased (reduces systematic error). The final vector becomes:

$$\phi(x_1, x_2, \dots, x_n)_i = \sum_{x_j \,:\, h(x_j) = i} \xi(x_j)$$

- $m$ is a hyperparameter; typically chosen as a power of 2 for fast modulo via bitmasking: $m = 2^k$.
- ***Hash functions:*** MurmurHash, FNV-1a, xxHash — chosen for speed and uniform distribution.
- The process is stateless and embarrassingly parallel — no global vocabulary object needed.

---

## **Challenges**

- ***Collisions:*** distinct features share a bucket → their signals interfere; hurts interpretability and model accuracy.
- ***Irreversibility:*** cannot recover the original feature name from its hash index — debugging is hard.
- ***No native OOV handling:*** a rare or unseen feature just hashes like any other — good or bad depending on context.
- ***Hyperparameter sensitivity:*** too small an $m$ → high collision rate; too large → wasted memory with sparse occupancy.
- ***Hash function matters:*** a poor hash (e.g. naive polynomial) causes clustering → collision rate spikes for similar strings.
- ***Feature interaction leakage:*** two different features that collide can appear correlated to the model even if they aren't.

---

## **Things to Keep in Mind**

- Start with $m = 2^{18}$ to $2^{22}$ (~250K–4M) for text features; tune down if memory-bound.
- Always use the sign hash trick alongside the main hash to reduce collision bias.
- Prefer well-tested hash functions (MurmurHash3, xxHash) over custom ones — uniform distribution is critical.
- ***Namespace prefixes:*** add a prefix to feature names (`"user_id=42"` vs `"item_id=42"`) to avoid cross-field collisions.
- Monitor collision rate empirically: hash the training set, count how many distinct features share each bucket.
- Works best with linear models and shallow trees.
- ***Standard implementations:*** Scikit-learn's `HashingVectorizer`.

In [2]:
import pandas as pd
import numpy as np

In [4]:
n_users = 1000
user_ids = np.arange(1, n_users + 1)

n_samples = 5000
sampled_user_ids = np.random.choice(user_ids, size=n_samples, replace=True)

data = pd.DataFrame({
    'user_id': sampled_user_ids,
    'click_through_rate': np.random.rand(n_samples)
})

data['user_id_str'] = 'user_id=' + data['user_id'].astype(str)
data.drop('user_id', axis=1, inplace=True)

data.head()

,click_through_rate,user_id_str
0,0.425760,user_id=83
1,0.342938,user_id=958
2,0.544564,user_id=429
3,0.344663,user_id=759
4,0.959918,user_id=166


In [11]:
from sklearn.feature_extraction import FeatureHasher

hasher = FeatureHasher(n_features=10, input_type='string', alternate_sign=True)
hashed_features = hasher.transform(data['user_id_str'].values.reshape(-1, 1))

hashed_df = pd.DataFrame(hashed_features.toarray(), columns=[f'feature_{i}' for i in range(10)])
hashed_df['click_through_rate'] = data['click_through_rate']
hashed_df.head()

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,click_through_rate
0,0.0,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.425760
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.342938
2,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.544564
3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.344663
4,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.959918


## **Comparison**
- ***Model type:*** OHE and Binary are safe for linear and distance-based models since they preserve metric meaning. Tree-based and gradient boosting models handle Ordinal and Target encoding well — they don't care about metric spacing. Target encoding is particularly powerful with gradient boosting (XGBoost, LightGBM).
- ***Cardinality:*** OHE blows up your feature space beyond ~15–20 categories — switch to Binary or Target at that point. Hashing is the only encoder that stays bounded regardless of how many unique values exist, making it the only real choice for very high cardinality or unknown vocabularies.
- ***Streaming / online learning:*** Hashing and Ordinal are the only safe options since they require no fit-time statistics. Target, Binary, and OHE all require a full pass over training data before they can encode anything.